<a href="https://colab.research.google.com/github/dhkdsns20/Capstone-Design/blob/main/showmap_dijkstra_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Path Finding Algorithm
## Find strongest path with thpt

In [1]:
pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 1.9 MB/s eta 0:00:00


In [ ]:
# from pyrosm import OSM
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import folium
import heapq

#  출발지 및 목적지 노드 설정 함수 정의
def get_nearest_node(G, latitude, longitude):
    """
    입력된 위도와 경도에 가장 가까운 그래프 노드를 찾는 함수.

    - G: 도로 네트워크 그래프
    - latitude: 위도
    - longitude: 경도

    반환: 가장 가까운 노드 ID
    """
    return ox.distance.nearest_nodes(G, longitude, latitude)

# 최단 경로 탐색 함수 정의
def find_shortest_path(G, origin_point, destination_point):
    """
    입력된 출발지와 목적지 간 최단 경로를 찾는 함수.

    - G: 도로 네트워크 그래프
    - origin_point: 출발지 좌표 (위도, 경도)
    - destination_point: 목적지 좌표 (위도, 경도)

    반환: 최단 경로 노드 리스트
    """
    # # 출발지와 목적지의 가장 가까운 노드 찾기
    # origin_node = get_nearest_node(G, *origin_point)
    # destination_node = get_nearest_node(G, *destination_point)

    # 최단 경로 계산 (Dijkstra 알고리즘)
    shortest_path = nx.shortest_path(G, origin_point, destination_point, weight='length',method="dijkstra") #
    return shortest_path


def plot_route(G, path):
    """
    최단 경로를 그래프에 시각화하는 함수.

    - G: 도로 네트워크 그래프
    - path: 최단 경로 노드 리스트
    """
    if path is None or len(path) == 0:
        print("경로가 유효하지 않아 시각화를 생략합니다.")
        return

    try:
        fig, ax = ox.plot_graph_route(G, path, route_linewidth=2, node_size=0, bgcolor='k')
        plt.show()
    except Exception as e:
        print(f"경로 시각화 중 오류 발생: {e}")


# 도로만 남기는 함수
def create_filtered_graph(G, excluded_highway_types):
    """
    특정 highway 유형을 제외한 새로운 그래프를 반환하는 함수.

    - G: NetworkX 그래프
    - excluded_highway_types: 제외할 highway 유형 리스트 (예: ['footway'])

    반환: 제외된 엣지가 제거된 새로운 NetworkX 그래프
    """
    # 그래프 복사 (원본을 유지하기 위해)
    filtered_graph = G.copy()

    # 제거 대상 엣지 식별 및 제거
    edges_to_remove = [
        (u, v) for u, v, data in filtered_graph.edges(data=True)
        if data.get('highway') in excluded_highway_types
    ]
    filtered_graph.remove_edges_from(edges_to_remove)

    # print(f"Removed {len(edges_to_remove)} edges with highway types: {excluded_highway_types}")
    return filtered_graph

# 도로 네트워크 시각화 함수 정의 (Node ID 표시 추가) - 추가기능
def plot_map_with_node_ids(G, poi_coords=None):
    """
    도로 네트워크와 각 노드의 ID를 시각화하는 함수.

    - G: 도로 네트워크 그래프
    - poi_coords: POI 위치 좌표 리스트 (선택적)
    """
    # 도로 네트워크 시각화
    fig, ax = ox.plot_graph(G, show=False, close=False)

    i = 1
    # 각 노드의 위치에 Node ID를 표시
    for node, data in G.nodes(data=True):
        x, y = data['x'], data['y']
        if node == 10666146667 or node == 9274780719:
            ax.text(x, y, str(node), fontsize=8, color='blue')  # Node ID 표시

    # POI 위치 시각화 (있을 경우)
    if poi_coords:
        y, x = zip(*poi_coords)
        ax.scatter(x, y, c='red', s=30, label='POI')
        ax.legend()

    plt.show()

# throughput 데이터를 노드에 매핑하는 함수 정의
def map_throughput_to_nodes(G, throughput_excel):
    """
    throughput 데이터를 그래프 노드 속성으로 매핑하는 함수.

    - G: 도로 네트워크 그래프
    - throughput_excel: 노드 ID와 throughput 값을 가진 DataFrame

    반환: throughput 속성이 추가된 그래프
    """
    # 노드 ID를 문자열로 변환하여 통일
    throughput_excel['Node ID'] = throughput_excel['Node ID'].astype(str)

    for node, data in G.nodes(data=True):
        # 그래프 노드 ID도 문자열로 변환
        node_str = str(node)
        # 노드 ID와 일치하는 Throughput 값을 찾음
        throughput_value = throughput_excel.loc[throughput_excel['Node ID'] == node_str, 'Throughput(Mbps)'].values
        if len(throughput_value) > 0:
            G.nodes[node]['throughput'] = throughput_value[0]  # Throughput 값을 노드에 추가
        else:
            G.nodes[node]['throughput'] = 0  # 데이터가 없을 경우 0으로 설정
    return G


# thpt을 가중치로 하여 신호 세기가 강한 경로를 찾는 알고리즘 함수 생성

In [ ]:
# 누적 가중치를 고려한 함수
def find_strongest_path(G, source, target):
    """
    Dijkstra 알고리즘을 수정하여 누적 가중치를 고려한 경로를 찾는 함수.

    - G: NetworkX 그래프
    - source: 출발 노드
    - target: 도착 노드

    반환:
    - 최단 경로의 노드 리스트
    - 경로의 누적 가중치
    """
    # 우선순위 큐 초기화 (누적 가중치 기준)
    queue = [(0, source, [])]  # (누적 가중치, 현재 노드, 경로)
    visited = set()

    while queue:
        cum_weight, current_node, path = heapq.heappop(queue)

        # 이미 방문한 노드는 무시
        if current_node in visited:
            continue
        visited.add(current_node)

        # 현재 노드를 경로에 추가
        path = path + [current_node]

        # 목표 노드에 도달하면 결과 반환
        if current_node == target:
            return path, cum_weight

        # 인접 노드 탐색
        for neighbor in G.neighbors(current_node):
            edge_data = G[current_node][neighbor]

            # ✅ 단일 간선 딕셔너리 접근 — Graph 타입에서는 이렇게 직접 접근해야 함
            edge_weight = edge_data.get('weight', float('inf'))

            # ✅ 유효한 가중치일 경우에만 큐에 추가
            if edge_weight < float('inf'):
                new_cum_weight = cum_weight + edge_weight
                heapq.heappush(queue, (new_cum_weight, neighbor, path))

            # # 최소 가중치 선택 (다중 간선 고려)
            # min_edge_weight = float('inf')
            # for key, data in edge_data.items():
            #     edge_weight = data.get('weight', float('inf'))
            #     if edge_weight is not None:
            #         min_edge_weight = min(min_edge_weight, edge_weight)

            # # 최소 가중치를 기반으로 경로 확장
            # if min_edge_weight < float('inf'):
            #     new_cum_weight = cum_weight + min_edge_weight
            #     heapq.heappush(queue, (new_cum_weight, neighbor, path))

    return None, float('inf')




# 가중치 계산 함수
def calculate_weight(throughput, distance, ratio):
    """
    - Throughput : Throughput 값
    - distance   : distance 값
    - ratio      : Throughput과 거리의 비율 (0~1)

    """
    # 최종 가중치 계산
    weight = (ratio * throughput) + ((1 - ratio) * distance)
    return weight



# 간선 가중치 설정 함수
def set_edge_weights_based_on_ratio(G, ratio):
    """
    간선의 가중치를 두 노드의 throughput 값과 거리 값의 비율로 설정하는 함수.
    - G: 도로 네트워크 그래프
    - ratio: Throughput:Distance 비율 (0~1)
    """

    for u, v, data in G.edges(data=True):
        # 노드의 throughput 값을 가져옴 (없으면 0으로 설정)
        throughput_u = G.nodes[u].get('throughput', 0)
        throughput_v = G.nodes[v].get('throughput', 0)
        distance = data.get('length', float('inf'))

        # 두 노드의 throughput 평균 계산
        avg_throughput = (throughput_u + throughput_v) / 2
        weight = calculate_weight(avg_throughput, distance, ratio)
        data['weight'] = weight
    return G


- 충북대 osm 불러오기 및 src, target 지정

In [ ]:
# 출발지 및 목적지 좌표 설정
origin = (36.6256453, 127.4314837)        # 청주 시외버스터미널
destination = (36.6246818, 127.4638686)   # 충북대학교 병원

# 중심 좌표 계산 (중간 지점)
center_lat = (origin[0] + destination[0]) / 2
center_lon = (origin[1] + destination[1]) / 2
center_point = (center_lat, center_lon)

# 반경 2km 도로 네트워크 불러오기
radius = 2000  # 반경 2km
cheongju_graph = ox.graph_from_point(center_point, dist=radius, network_type="walk")

# 출발지와 목적지의 가장 가까운 노드 찾기
origin_node = get_nearest_node(cheongju_graph, *origin)
destination_node = get_nearest_node(cheongju_graph, *destination)

print(f"출발지 노드 ID (청주 시외버스터미널): {origin_node}")
print(f"목적지 노드 ID (충북대학교 병원): {destination_node}")

# 시각화로 반경 확인
fig, ax = ox.plot_graph(cheongju_graph, bgcolor='white', node_size=10, edge_linewidth=1)

In [ ]:
## 도보 제외한 네트워크 Graph 생성
# 제외할 highway 유형 설정
excluded_highways = ['footway']

# 필터링된 그래프 생성
filtered_graph = create_filtered_graph(cheongju_graph, excluded_highways)

In [ ]:
plot_map_with_node_ids(filtered_graph)

- rxsite 노드 ID 반환 코드

## 유효 노드 (교차로 사이 노드 포함 : 도로생성을 위해서)

In [ ]:
# 단계 1 : 기본 유효 노드 집합 생성
# Rx 노드 ID 불러오기
rx_file_path = 'rxsite_throughput_with_node_id.xlsx'
rx_data = pd.read_excel(rx_file_path)
rx_node_ids = rx_data['Node ID'].tolist()

# 출발지 및 목적지 좌표 설정
origin = (36.6256453, 127.4314837)        # 청주 시외버스터미널
destination = (36.6246818, 127.4638686)   # 충북대학교 병원

# # OSM 데이터 로드
# cheongju_graph = ox.graph_from_place('Cheongju, South Korea', network_type='drive')

# 출발지와 목적지의 가장 가까운 노드 찾기
origin_node = ox.distance.nearest_nodes(cheongju_graph, origin[1], origin[0])
destination_node = ox.distance.nearest_nodes(cheongju_graph, destination[1], destination[0])

# 출발지, 목적지, Rx 노드 ID 합치기
valid_node_ids = set(rx_node_ids + [origin_node, destination_node])

# Rx 노드만 포함된 독립 그래프 생성
rx_graph = nx.Graph()
for _, row in rx_data.iterrows():
    node_id = row['Node ID']
    lat, lon = row['RxLatitude'], row['RxLongitude']
    rx_graph.add_node(node_id, x=lon, y=lat)

# 출발지, 목적지, Rx 노드만 포함하는 필터링된 그래프 생성
node_filtered_graph = cheongju_graph.subgraph(valid_node_ids).copy()

# 유효 노드들로 구성된 도로 찾기
valid_edges = []
for u, v, data in node_filtered_graph.edges(data=True):
    if u in rx_node_ids and v in rx_node_ids:  # Rx 노드 간 연결만 포함
        valid_edges.append((u, v))

# 유효 노드와 도로 시각화 개선
valid_graph = nx.Graph()
valid_graph.add_edges_from(valid_edges)
valid_graph.add_nodes_from(valid_node_ids)

# 빠른 시각화 함수
def plot_fast_graph(G, rx_graph, origin_node, destination_node):
    fig, ax = plt.subplots(figsize=(10,10))
    ox.plot_graph(G, ax=ax, node_size=5, node_color='white', edge_color='gray', edge_linewidth=0.8, show=False, close=False, bgcolor='black')

    # Rx 노드 시각화
    for node in rx_graph.nodes:
        x, y = rx_graph.nodes[node]['x'], rx_graph.nodes[node]['y']
        ax.scatter(x, y, c='blue', s=50, marker='x')
        ax.text(x + 0.0001, y + 0.0001, str(node), fontsize=8, color='blue')

    # 출발지 노드 표시
    x, y = G.nodes[origin_node]['x'], G.nodes[origin_node]['y']
    ax.scatter(x, y, c='red', s=100, label='Origin', marker='o', edgecolors='white', linewidths=2)
    ax.text(x, y, f'Origin\n{origin_node}', fontsize=10, color='red', ha='left', va='bottom')

    # 목적지 노드 표시
    x, y = G.nodes[destination_node]['x'], G.nodes[destination_node]['y']
    ax.scatter(x, y, c='green', s=100, label='Destination', marker='o', edgecolors='white', linewidths=2)
    ax.text(x, y, f'Destination\n{destination_node}', fontsize=10, color='green', ha='left', va='bottom')

    plt.title("도로 네트워크와 Rx 노드 시각화 (OSMnx + Matplotlib)", color='white', fontsize=14)
    plt.show()

# 시각화 실행
plot_fast_graph(cheongju_graph, rx_graph, origin_node, destination_node)


In [ ]:
# 출발지와 도착지를 별도로 관리
origin_destination_ids = {origin_node, destination_node}

# Rx 노드 (rx_graph 노드) 집합화
rx_only_node_ids = set(rx_graph.nodes)

# 전체 유효 노드
valid_node_ids = rx_only_node_ids | origin_destination_ids  # 합집합

# 출력
print(f"✅ Rx 노드 수: {len(rx_only_node_ids)}개")
print(f"✅ 출발지, 도착지 노드 수: {len(origin_destination_ids)}개")
print(f"✅ 전체 유효 노드 수 (Rx + 출발지/도착지): {len(valid_node_ids)}개")


In [ ]:
def save_only_rx_nodes_with_coordinates(rx_graph, output_path):
    """
    Rx 노드만 Node ID + 위도(latitude) + 경도(longitude)를 엑셀로 저장하는 함수
    (출발지/도착지 노드는 저장하지 않음)
    """
    records = []

    # Rx 노드만 저장 (rx_graph 기준)
    for node in rx_graph.nodes:
        x = rx_graph.nodes[node].get('x', None)
        y = rx_graph.nodes[node].get('y', None)
        if x is not None and y is not None:
            records.append({'Node ID': node, 'latitude': y, 'longitude': x})

    # 데이터프레임으로 변환 및 엑셀 저장
    df = pd.DataFrame(records)
    df.to_excel(output_path, index=False)
    print(f"✅ 총 {len(records)}개 Rx 노드를 '{output_path}'에 저장했습니다.")

In [ ]:
# 저장 (출발지,목적지 총 2개의 노드는 제외)
save_only_rx_nodes_with_coordinates(
    rx_graph=rx_graph,
    output_path='only_rx_nodes_with_coordinates.xlsx'
)


- Rx 노드 사이 도로 시각화

In [ ]:
# 출발지/도착지 제거된 Rx 노드만 포함
valid_node_ids = set(rx_node_ids)  # Rx 노드만

# 유효 노드들로 구성된 도로 찾기 (Rx 노드 간 최단 경로 포함)
valid_edges = []
for i in range(len(rx_node_ids)):
    for j in range(i + 1, len(rx_node_ids)):
        u, v = rx_node_ids[i], rx_node_ids[j]
        try:
            # Rx 노드 간 최단 경로 계산
            path = nx.shortest_path(cheongju_graph, source=u, target=v, weight='length')

            # 경로 상의 모든 노드를 추가
            for k in range(len(path) - 1):
                valid_edges.append((path[k], path[k+1]))
                valid_node_ids.add(path[k])
                valid_node_ids.add(path[k+1])
        except nx.NetworkXNoPath:
            # 경로가 없는 경우 무시
            continue

# 유효 그래프 재구성
valid_graph = nx.Graph()
valid_graph.add_edges_from(valid_edges)
valid_graph.add_nodes_from(valid_node_ids)


# 시각화 함수 (Rx 노드 간 도로 포함)
def plot_fast_graph(G, rx_graph, origin_node, destination_node):
    fig, ax = plt.subplots(figsize=(12, 12))
    ox.plot_graph(G, ax=ax, node_size=5, node_color='white', edge_color='gray', edge_linewidth=0.8, show=False, close=False, bgcolor='black')

    # Rx 노드 시각화
    for node in rx_graph.nodes:
        x, y = rx_graph.nodes[node]['x'], rx_graph.nodes[node]['y']
        ax.scatter(x, y, c='blue', s=50, marker='x')
        ax.text(x + 0.0001, y + 0.0001, str(node), fontsize=8, color='blue')

    # 출발지 노드 표시
    x, y = G.nodes[origin_node]['x'], G.nodes[origin_node]['y']
    ax.scatter(x, y, c='red', s=100, label='Origin', marker='o', edgecolors='white', linewidths=2)
    ax.text(x, y, f'Origin\n{origin_node}', fontsize=10, color='red', ha='left', va='bottom')

    # 목적지 노드 표시
    x, y = G.nodes[destination_node]['x'], G.nodes[destination_node]['y']
    ax.scatter(x, y, c='green', s=100, label='Destination', marker='o', edgecolors='white', linewidths=2)
    ax.text(x, y, f'Destination\n{destination_node}', fontsize=10, color='green', ha='left', va='bottom')

    # Rx 노드 사이 도로 시각화
    for u, v in valid_edges:
        x1, y1 = G.nodes[u]['x'], G.nodes[u]['y']
        x2, y2 = G.nodes[v]['x'], G.nodes[v]['y']
        ax.plot([x1, x2], [y1, y2], color='blue', linewidth=2)

    plt.title("도로 네트워크와 Rx 노드 연결 경로 시각화", color='white', fontsize=14)
    plt.show()

# 시각화 실행
plot_fast_graph(cheongju_graph, rx_graph, origin_node, destination_node)


- 유효 간선 시각화

In [ ]:
print(f"✅ 생성된 유효 간선 수: {len(valid_edges)}개")

In [ ]:
def visualize_trunks(G, valid_edges, title="Trunks Visualization"):
    """
    G: 노드들의 x, y 좌표를 가지고 있는 그래프 (cheongju_graph 또는 valid_graph)
    valid_edges: trunk 리스트 ([(u, v), (u, v), ...])
    title: 플롯 제목
    """

    # 좌표 가져오기
    pos = {node: (G.nodes[node]['x'], G.nodes[node]['y']) for node in G.nodes if 'x' in G.nodes[node] and 'y' in G.nodes[node]}

    # 유효한 trunk만 선택 (양쪽 노드 모두 pos에 있는 경우만)
    filtered_edges = [(u, v) for u, v in valid_edges if u in pos and v in pos]

    plt.figure(figsize=(14, 14))

    # trunk 그리기
    for u, v in filtered_edges:
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        plt.plot([x1, x2], [y1, y2], color='blue', linewidth=0.8, alpha=0.7)

    # 노드 그리기
    x_nodes = [pos[n][0] for n in pos]
    y_nodes = [pos[n][1] for n in pos]
    plt.scatter(x_nodes, y_nodes, c='white', s=5, alpha=0.8, edgecolors='black')

    plt.title(title, fontsize=16)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.grid(True)
    plt.gca().set_facecolor('white')
    plt.show()

    return filtered_edges  # ✨ 추가: 필터링된 간선 반환


In [ ]:
# 호출할 때, 반환값으로 저장
filtered_valid_edges = visualize_trunks(cheongju_graph, valid_edges, title="유효 노드 간 Trunks 시각화")


- 간선이 유효노드를 잇고 있는지 확인

In [ ]:
def verify_filtered_edges_connection(filtered_edges, valid_node_ids):
    """
    filtered_edges: 필터링된 trunk 간선 리스트
    valid_node_ids: 연결이 잘 되어야 하는 유효 노드 리스트/집합
    """
    # 1. 필터링된 간선으로 그래프 생성
    G_check = nx.Graph()
    G_check.add_edges_from(filtered_edges)

    # 2. 연결 컴포넌트 찾기
    components = list(nx.connected_components(G_check))

    # 3. 유효 노드들이 같은 컴포넌트에 있는지 확인
    for comp in components:
        if set(valid_node_ids).issubset(comp):
            print(f"✅ 모든 {len(valid_node_ids)}개의 유효 노드가 하나의 연결 컴포넌트 안에 있습니다.")
            return True

    print(f"❌ 유효 노드들이 하나의 연결 컴포넌트에 있지 않습니다.")
    return False

In [ ]:
# 예시
verify_filtered_edges_connection(filtered_valid_edges, rx_node_ids)


### Throughput 매칭
- thpt dataset 불러오기

In [ ]:
# 엑셀 comm data(throughput) 불러오기
throughput_excel = pd.read_excel('rxsite_throughput_with_node_id.xlsx')

# throughput 정보를 노드에 매핑
G_set_node = map_throughput_to_nodes(valid_graph, throughput_excel)

- mapping 확인

In [ ]:
def print_throughput_of_valid_nodes(G, rx_node_ids):
    """
    G: throughput이 매핑된 그래프 (G_set_node)
    rx_node_ids: Rx 유효 노드 리스트 또는 집합
    유효 노드들의 처리율(throughput) 값을 출력하는 함수
    """
    count_found = 0
    count_missing = 0

    for node in rx_node_ids:
        if node in G.nodes:
            throughput = G.nodes[node].get('throughput', None)
            if throughput is not None:
                print(f"Node {node}: throughput = {throughput}")
                count_found += 1
            else:
                print(f"❌ Node {node}: throughput 정보 없음")
                count_missing += 1
        else:
            print(f"❌ Node {node}: 그래프에 존재하지 않음")
            count_missing += 1

    print(f"\n✅ 처리율 정보 있는 노드 수: {count_found}개")
    print(f"❌ 처리율 정보 없는 노드 수: {count_missing}개")


In [ ]:
print_throughput_of_valid_nodes(G_set_node, rx_node_ids)